# Computer Vision — โจทย์แบบ "จำแนกรูปภาพ" (Image Classification)

รูป 1 รูป -> 1 ป้าย (เช่น บ้าน/ไม่ใช่บ้าน, สุนัข/แมว, ปกติ/ผิดปกติ, ชนิดสินค้า)

วิธีในโน้ตบุ๊กนี้ (ทำจากบนลงล่าง):
- วิธีที่ 1 (ง่ายสุด แนะนำมือใหม่ทำแค่นี้) = `AutoGluon` กดรันแล้วมันเลือกโมเดล+เทรนให้เอง
- วิธีที่ 2 (ไม่บังคับ ทำถ้าอยากได้คะแนนสูงขึ้น) = `timm` เลือกโมเดลเอง + TTA


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ: input เป็นรูป และต้องตอบว่า "รูปนี้คือคลาสอะไร" (ตอบเป็นป้าย/ตัวเลข ไม่ใช่ข้อความยาว)
ถ้าโจทย์ให้ "หากรอบวัตถุ (กล่อง)" -> ไปใช้ `object_detection.ipynb`
ถ้าโจทย์ให้ "บรรยายรูปเป็นข้อความ" -> ไปหัวข้อ 03 Multimodal

ต้องแก้อะไรบ้าง (หาคำว่า `# << แก้`): ชื่อ competition, path โฟลเดอร์รูป, ชื่อคอลัมน์, จำนวนคลาส

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U "autogluon.multimodal"        # วิธีที่ 1
!pip -q install timm torch torchvision pillow scikit-learn pandas numpy tqdm   # วิธีที่ 2

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "super-ai-engineer-season-6-individual-hackathon-house-recognition"      # << แก้ตรงนี้: slug ของ competition (คือส่วนท้าย URL หลังคำว่า /competitions/)
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) คีย์ยาว ๆ จาก kaggle.json

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — ตั้งค่า path + CONFIG (ดูจาก output ด้านบนแล้วแก้ให้ตรง)

In [ ]:
import os, glob, pandas as pd, numpy as np

def find(name):   # หาไฟล์ตามชื่อ
    h = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True); return h[0] if h else None
def img_dir(keyword):   # หาโฟลเดอร์รูปที่ชื่อมี keyword (train/test)
    cand = [d for d,_,fs in os.walk(DATA_DIR)
            if keyword in d.lower() and any(f.lower().endswith((".jpg",".png",".jpeg")) for f in fs)]
    return max(cand, key=lambda d: len(os.listdir(d))) if cand else None

TRAIN_CSV  = find("train.csv")               # << แก้ถ้าหาไม่เจอ
SAMPLE_SUB = find("sample_submission.csv")
TRAIN_IMG  = img_dir("train")                # << แก้ถ้าผิด (โฟลเดอร์รูป train)
TEST_IMG   = img_dir("test")                 # << แก้ถ้าผิด (โฟลเดอร์รูป test)

df = pd.read_csv(TRAIN_CSV); sample = pd.read_csv(SAMPLE_SUB)
print("train.csv คอลัมน์:", list(df.columns)); print("sample คอลัมน์:", list(sample.columns))
print("รูป train:", TRAIN_IMG, "| รูป test:", TEST_IMG)
display(df.head()); display(sample.head())

IMG_COL    = "image_name"        # << แก้: คอลัมน์ชื่อไฟล์รูปใน train.csv
LABEL_COL  = "class"             # << แก้: คอลัมน์ป้ายใน train.csv
ID_COL     = sample.columns[0]   # คอลัมน์ id ใน sample (อัตโนมัติ)
ANSWER_COL = sample.columns[1]   # คอลัมน์คำตอบใน sample (อัตโนมัติ)
TEST_EXT   = ".jpg"              # << แก้ถ้านามสกุลไฟล์ test ไม่ใช่ .jpg
NUM_CLASSES = int(df[LABEL_COL].nunique())
print("จำนวนคลาส =", NUM_CLASSES)

## วิธีที่ 1 — AutoGluon (ง่ายสุด มือใหม่ทำแค่นี้ก็ได้คะแนนดี)

แค่ทำตารางที่มี path รูป + ป้าย แล้วสั่ง fit เดี๋ยว AutoGluon เลือกโมเดล + เทรน + รวมโมเดลให้เอง
ปรับ `time_limit` (วินาที) ตามเวลาที่มี

In [ ]:
from autogluon.multimodal import MultiModalPredictor

train_df = df.copy()
train_df["image"] = train_df[IMG_COL].apply(lambda n: os.path.join(TRAIN_IMG, str(n)))
predictor = MultiModalPredictor(label=LABEL_COL, eval_metric="accuracy", path="ag_imgcls")
predictor.fit(train_df[["image", LABEL_COL]], time_limit=900)   # << แก้: 900 วิ = 15 นาที

test_df = sample.copy()
test_df["image"] = test_df[ID_COL].apply(lambda i: os.path.join(TEST_IMG, str(i)+TEST_EXT))
out = sample.copy(); out[ANSWER_COL] = np.asarray(predictor.predict(test_df[["image"]])).astype(int)
out.to_csv("submission.csv", index=False)
print("บันทึก submission.csv เรียบร้อย"); display(out.head())

## วิธีที่ 2 — timm fine-tune + TTA (ไม่บังคับ ทำถ้าอยากได้คะแนนสูงขึ้น ต้องมี GPU)

โหลดโมเดล pretrained จาก `timm` มาเทรนต่อ + augmentation + TTA
เลือกโมเดลอื่นได้ที่ `MODEL_NAME` (ดูตัวเลือกใน reference_cheatsheet.md)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from PIL import Image
from tqdm.auto import tqdm

MODEL_NAME="tf_efficientnetv2_s.in21k_ft_in1k"  # << แก้โมเดลได้
IMG_SIZE=300; EPOCHS=12; BATCH=32; LR=3e-4      # << แก้ตามเวลา/GPU
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; AMP=(DEVICE=="cuda")
MEAN,STD=[0.485,0.456,0.406],[0.229,0.224,0.225]
tf_tr=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(.3,.3,.3,.1),transforms.RandomRotation(15),transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),transforms.RandomErasing(p=0.2)])
tf_ev=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
tf_tta=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
class DS(Dataset):
    def __init__(s,frame,d,tf,test=False,idc=None): s.f=frame.reset_index(drop=True);s.d=d;s.tf=tf;s.t=test;s.idc=idc
    def __len__(s): return len(s.f)
    def __getitem__(s,i):
        r=s.f.iloc[i]
        if s.t: return s.tf(Image.open(os.path.join(s.d,str(r[s.idc])+TEST_EXT)).convert("RGB")),0
        return s.tf(Image.open(os.path.join(s.d,str(r[IMG_COL]))).convert("RGB")),int(r[LABEL_COL])
skf=StratifiedKFold(5,shuffle=True,random_state=42)
tr,va=next(iter(skf.split(df,df[LABEL_COL])))
model=timm.create_model(MODEL_NAME,pretrained=True,num_classes=NUM_CLASSES,drop_rate=0.3).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
crit=nn.CrossEntropyLoss(label_smoothing=0.1); scaler=torch.cuda.amp.GradScaler(enabled=AMP)
dl=DataLoader(DS(df.iloc[tr],TRAIN_IMG,tf_tr),BATCH,shuffle=True,num_workers=2,drop_last=True)
vl=DataLoader(DS(df.iloc[va],TRAIN_IMG,tf_ev),BATCH,num_workers=2)
for ep in range(EPOCHS):
    model.train()
    for x,y in tqdm(dl,desc=f"ep{ep+1}/{EPOCHS}",leave=False):
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=AMP): loss=crit(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sch.step()
    model.eval(); cc=tt=0
    with torch.no_grad():
        for x,y in vl:
            with torch.cuda.amp.autocast(enabled=AMP): pr=model(x.to(DEVICE)).argmax(1).cpu()
            cc+=(pr==y).sum().item(); tt+=len(y)
    print(f"ep{ep+1} val_acc={cc/tt:.4f}")
probs=np.zeros((len(sample),NUM_CLASSES))
for s_ in range(5):
    tf=tf_ev if s_==0 else tf_tta
    dl=DataLoader(DS(sample,TEST_IMG,tf,test=True,idc=ID_COL),BATCH,num_workers=2)
    o=[]
    with torch.no_grad():
        for x,_ in tqdm(dl,desc=f"TTA{s_+1}",leave=False):
            with torch.cuda.amp.autocast(enabled=AMP): o.append(F.softmax(model(x.to(DEVICE)),1).float().cpu().numpy())
    probs+=np.vstack(o)
out=sample.copy(); out[ANSWER_COL]=probs.argmax(1).astype(int)
out.to_csv("submission_timm.csv",index=False); print("บันทึก submission_timm.csv"); display(out.head())

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "image classification"
!kaggle competitions submissions -c {COMP} | head